# AGENTIC AI
## SESSION 15

Assignment questions and answers


## Question 1

Install LangChain and create a simple Python script that uses LangChain to summarize a Flipkart product review. Display the original review and the summary in the console.

**Answer:**

**Installation:**
```bash
pip install langchain langchain-openai
```

**Python Script:**
```python
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Set up the model (requires OPENAI_API_KEY environment variable)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Flipkart product review
review = """
I bought this boat Rockerz 450 headphone two weeks ago. The sound quality is 
exceptional for this price range with deep bass. Battery easily lasts 15 hours. 
However, after wearing it for more than an hour, my ears start hurting because 
the cushion padding is a bit hard. Overall, a great budget purchase.
"""

# Prompt template
template = "Summarize the following product review in one sentence:\n\n{review}"
prompt = PromptTemplate(template=template, input_variables=["review"])

# Run chain
chain = prompt | llm
summary = chain.invoke({"review": review})

# Display results
print("Original Review:\n", review.strip())
print("\nSummary:\n", summary.content)
```

## Question 2

Use LangChain to build a conversational agent that answers questions about IPL cricket teams using a provided list of team names and captains. The agent should respond accurately to queries like 'Who is the captain of Mumbai Indians?'.

**Answer:**

```python
import os
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# 1. Dataset
ipl_captains = {
    "Mumbai Indians": "Hardik Pandya",
    "Chennai Super Kings": "Ruturaj Gaikwad",
    "Royal Challengers Bengaluru": "Faf du Plessis",
    "Kolkata Knight Riders": "Shreyas Iyer",
    "Delhi Capitals": "Rishabh Pant",
    "Rajasthan Royals": "Sanju Samson",
    "Sunrisers Hyderabad": "Pat Cummins",
    "Lucknow Super Giants": "KL Rahul",
    "Gujarat Titans": "Shubman Gill",
    "Punjab Kings": "Shikhar Dhawan",
}


# 2. Tool to query captain data
@tool
def get_ipl_captain(team_name: str) -> str:
    """Finds the captain for a given IPL team."""
    for team, captain in ipl_captains.items():
        if team.lower() == team_name.lower():
            return f"The captain of {team} is {captain}."
    return f"Team '{team_name}' not found."


# 3. Model, Prompt, and Agent Setup
tools = [get_ipl_captain]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant answering IPL questions using the provided tools.",
        ),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False)

# 4. Execution
response = agent_executor.invoke(
    {"input": "Who is the captain of Mumbai Indians?"}
)
print(response["output"])
```

## Question 3

Integrate LangGraph to design a flow where a user can ask for Zomato restaurant recommendations, and the agent first asks for the user's city, then suggests 3 restaurants from a hardcoded list based on the city.

**Answer:**

```python
from typing import TypedDict
from langgraph.graph import END, StateGraph


# Define state
class AgentState(TypedDict):
    city: str
    recommendations: list[str]


# Hardcoded restaurant data
RESTAURANTS = {
    "mumbai": ["Trishna", "Britannia & Co.", "The Bombay Canteen"],
    "delhi": ["Bukhara", "Indian Accent", "Karim's"],
    "bangalore": ["Toit", "Vidyarthi Bhavan", "Truffles"],
}


# Node: Ask for city
def ask_city(state: AgentState):
    city_input = input("Which city are you looking for restaurants in? ").strip()
    return {"city": city_input}


# Node: Suggest restaurants
def suggest_restaurants(state: AgentState):
    city = state["city"].lower()
    results = RESTAURANTS.get(
        city, ["Generic Bistro", "Central Cafe", "City Diner"]
    )
    print(f"Top 3 recommendations in {state['city']}:")
    for r in results:
        print(f"- {r}")
    return {"recommendations": results}


# Build workflow
workflow = StateGraph(AgentState)

workflow.add_node("ask_city", ask_city)
workflow.add_node("suggest_restaurants", suggest_restaurants)

workflow.set_entry_point("ask_city")
workflow.add_edge("ask_city", "suggest_restaurants")
workflow.add_edge("suggest_restaurants", END)

# Compile and run
app = workflow.compile()
app.invoke({"city": "", "recommendations": []})
```

## Question 4

Set up a basic MCP (Multi-Component Pipeline) workflow that chains together LangChain and LangGraph: first, summarize a BookMyShow movie review using LangChain, then use LangGraph to ask the user if they want more details about the movie, and display a follow-up message based on their response.

Hint: Use LangChain for the summarization step and LangGraph to manage the user interaction flow.

**Answer:**

```python
from typing import TypedDict
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph


# 1. State Definition
class MovieState(TypedDict):
    review: str
    summary: str
    user_choice: str
    message: str


# 2. LangChain Summarization Step
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = PromptTemplate.from_template("Summarize this movie review briefly:\n{review}")
summarize_chain = prompt | llm


def summarize_node(state: MovieState) -> dict:
    summary = summarize_chain.invoke({"review": state["review"]}).content
    return {"summary": summary}


# 3. LangGraph Interactive Steps
def ask_user_node(state: MovieState) -> dict:
    print(f"\nReview Summary: {state['summary']}")
    choice = (
        input("\nWould you like more details about the movie? (yes/no): ")
        .strip()
        .lower()
    )
    return {"user_choice": choice}


def provide_details_node(state: MovieState) -> dict:
    return {
        "message": "Here are more details: The movie is currently trending with an 8.5/10 rating on BookMyShow!"
    }


def end_flow_node(state: MovieState) -> dict:
    return {"message": "Thank you! Enjoy your movie browsing."}


def route_decision(state: MovieState) -> str:
    return "details" if state["user_choice"] == "yes" else "end"


# 4. Build and Compile Workflow
workflow = StateGraph(MovieState)

workflow.add_node("summarize", summarize_node)
workflow.add_node("ask_user", ask_user_node)
workflow.add_node("details", provide_details_node)
workflow.add_node("end", end_flow_node)

workflow.set_entry_point("summarize")
workflow.add_edge("summarize", "ask_user")
workflow.add_conditional_edges(
    "ask_user", route_decision, {"details": "details", "end": "end"}
)
workflow.add_edge("details", END)
workflow.add_edge("end", END)

app = workflow.compile()

# Example Execution
initial_review = (
    "Inception on BookMyShow: Visually stunning masterpiece with a gripping plot. "
    "Leonardo DiCaprio delivers an unforgettable performance. A must-watch thriller."
)

result = app.invoke({"review": initial_review})
print(result["message"])
```

## Question 5

Use ChatGPT or Copilot to generate a LangChain code snippet that fetches and summarizes trending topics from a mock Twitter API, then test and modify the code to print the top 2 trends in your console.

Hint: Paste the AI's code and document any changes you made to get it working.

**Answer:**

### Generated Code (from AI)

```python
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_openai import OpenAI

# Mock Twitter API
mock_twitter_data = [
    {
        "topic": "#AIRevolution",
        "tweets": [
            "AI is changing the world!",
            "New LLM released today, mind-blowing results.",
        ],
    },
    {
        "topic": "#TechNews",
        "tweets": [
            "Semiconductor shortages easing up.",
            "Tech stocks rebound this quarter.",
        ],
    },
    {
        "topic": "#SpaceExploration",
        "tweets": [
            "New rover lands safely.",
            "Images from deep space transmitted.",
        ],
    },
]

llm = OpenAI(api_key="your-api-key")
prompt = PromptTemplate.from_template("Summarize these tweets: {tweets}")
chain = LLMChain(llm=llm, prompt=prompt)

for item in mock_twitter_data:
    summary = chain.run(item["tweets"])
    print(f"Topic: {item['topic']}\nSummary: {summary}\n")
```

---

### Modified Working Code

```python
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Mock Twitter data
mock_twitter_data = [
    {
        "topic": "#AIRevolution",
        "tweets": [
            "AI is changing the world!",
            "New LLM released today, mind-blowing results.",
        ],
    },
    {
        "topic": "#TechNews",
        "tweets": [
            "Semiconductor shortages easing up.",
            "Tech stocks rebound this quarter.",
        ],
    },
    {
        "topic": "#SpaceExploration",
        "tweets": [
            "New rover lands safely.",
            "Images from deep space transmitted.",
        ],
    },
]

# Modern LCEL syntax
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
prompt = PromptTemplate.from_template(
    "Provide a one-sentence summary of these tweets: {tweets}"
)
chain = prompt | llm | StrOutputParser()

# Filter to top 2 trends and print
top_2_trends = mock_twitter_data[:2]

for item in top_2_trends:
    summary = chain.invoke({"tweets": " ".join(item["tweets"])})
    print(f"Topic: {item['topic']}")
    print(f"Summary: {summary}\n")
```

---

### Documented Changes

* **Slice Data to Top 2:** Added `mock_twitter_data[:2]` to limit the loop strictly to the top two trending topics.
* **Modern LangChain Syntax:** Replaced the deprecated `LLMChain` and `chain.run()` with the modern LangChain Expression Language (`chain = prompt | llm | StrOutputParser()` and `chain.invoke()`).
* **Chat Model Update:** Replaced legacy `OpenAI` with `ChatOpenAI(model="gpt-3.5-turbo")`.
* **Input Formatting:** Joined the list of tweet strings with `" ".join(item["tweets"])` before passing it to the prompt.